# Retrieval Control

Given that interference acts at encoding, what retrieval-phase mechanisms modulate vulnerability?
Two control mechanisms in CMR provide graded protection for intentional recall:

1. **Starting-context reinstatement** (`start_drift_rate`) — at test onset, context is drifted toward the pre-experimental state, biasing retrieval away from recently encoded interference items. Scaling this up makes the retrieval probe less similar to the interference region and more similar to the film region.

2. **Choice sensitivity** (`mcf_sensitivity` / tau) — the exponent in the power-scale transformation of M\_CF activations. Higher tau sharpens the Luce choice rule, concentrating retrieval probability on strongly activated items (film items with strong M\_CF associations) and suppressing weakly activated competitors.

Each is swept in isolation, then a 2×2 summary examines their interaction.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from jax import random

from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root, generate_trial_mask, load_data
from jaxcmr.selective_interference import (
    Paradigm,
    compute_n_presented,
    prepare_sweep,
    run_sweep,
    configure_rates,
    sweep_rngs,
    film_recalled_stats,
    load_or_fit_params,
    make_factory,
    standard_remap,
    plot_interference_spc,
    light_to_dark_colors,
    plot_summary_dv,
    add_filler_boundary,
)

warnings.filterwarnings("ignore")

In [ ]:
# --- paradigm geometry ---
N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 100
SHOW_BREAK_IN_SPC = True
SHOW_FILLERS_IN_SPC = True

# --- default scale factors ---
DEFAULT_BREAK_DRIFT_SCALE = 1.0
DEFAULT_BREAK_MCF_SCALE = 1.0
DEFAULT_REMINDER_START_DRIFT_SCALE = 4.0
DEFAULT_REMINDER_DRIFT_SCALE = 0.3
DEFAULT_FILLER_DRIFT_SCALE = 1.0
DEFAULT_FILLER_MCF_SCALE = 1.0

# --- fitted parameter scales ---
PARAM_SCALES = {
    "stop_probability_scale": 0.57,
}

# --- fitting ---
SEED = 0
DATA_TAG = "HealeyKahana2014"
DATA_PATH = "data/HealeyKahana2014.h5"
TRIAL_QUERY = "data['listtype'] == -1"
MODEL_NAME = "WeirdCMRPosStop"
BEST_OF = 1
RUN_TAG = f"best_of_{BEST_OF}"
REDO_FITS = False

In [ ]:
# --- derived objects ---
paradigm = Paradigm(
    n_film=N_FILM, n_break=N_BREAK,
    n_interference=N_INTERFERENCE, n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
)

N_BREAK_SHOWN = N_BREAK if SHOW_BREAK_IN_SPC else 0
STD_PRESENTED = compute_n_presented(
    paradigm, show_break=SHOW_BREAK_IN_SPC, show_fillers=SHOW_FILLERS_IN_SPC,
)

project_root = Path(find_project_root())
fit_dir = Path("results/fits")
fit_dir.mkdir(parents=True, exist_ok=True)
fit_path = fit_dir / f"{DATA_TAG}_{MODEL_NAME}_{RUN_TAG}.json"

data = load_data(str(project_root / DATA_PATH))
trial_mask = generate_trial_mask(data, TRIAL_QUERY)
factory = make_factory()
rng = random.PRNGKey(SEED)

params, n_subjects = load_or_fit_params(
    fit_path, param_scales=PARAM_SCALES,
    data=data, trial_mask=trial_mask,
    model_factory=factory, redo_fits=REDO_FITS, best_of=BEST_OF,
)
print(f"{n_subjects} subjects")

In [ ]:
# Pre-cache scales
_pre_cache_scales = dict(
    break_drift_scale=DEFAULT_BREAK_DRIFT_SCALE,
    break_mcf_scale=DEFAULT_BREAK_MCF_SCALE,
    reminder_start_drift_scale=DEFAULT_REMINDER_START_DRIFT_SCALE,
    reminder_drift_scale=DEFAULT_REMINDER_DRIFT_SCALE,
)

std_prep = prepare_sweep(
    params, paradigm, factory, cache_after="reminder",
    **_pre_cache_scales,
)
print(f"Standard tier cached: {n_subjects} subjects (list_length={paradigm.list_length})")

## Sweeping start\_drift\_rate scale

At test onset, `start_retrieving()` integrates the pre-experimental context state with drift rate `start_drift_rate`.
Scaling this up biases the retrieval probe away from the interference region.
At very high scales, context is dominated by pre-experimental features, making interference items contextually invisible.
We sweep the scale from 0.2 to 3.0 while holding tau at its fitted value.

In [ ]:
start_drift_scale_values = np.linspace(0.2, 3.0, 10)
sds_recalls_4d, rng = run_sweep(
    std_prep, rng,
    start_drift_scale=start_drift_scale_values,
    filler_drift_scale=DEFAULT_FILLER_DRIFT_SCALE,
    filler_mcf_scale=DEFAULT_FILLER_MCF_SCALE,
)

sds_spcs, sds_stats = [], []
for i in range(len(start_drift_scale_values)):
    recalls = sds_recalls_4d[i].reshape(-1, sds_recalls_4d.shape[-1])
    recalls = standard_remap(recalls, paradigm)
    sds_spcs.append(fixed_pres_spc(recalls, STD_PRESENTED))
    sds_stats.append(film_recalled_stats(recalls, paradigm, n_subjects))

print(f"Start drift scale sweep done: {start_drift_scale_values.round(2)}")

In [ ]:
labels = ["{:.2f}".format(v) for v in start_drift_scale_values]
colors = light_to_dark_colors(len(sds_spcs))
means, ci_lo, ci_hi = zip(*sds_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_interference_spc(
    sds_spcs, labels, n_film=N_FILM, n_break=N_BREAK_SHOWN,
    n_presented=STD_PRESENTED, color_cycle=colors, axis=axes[0],
)
if SHOW_FILLERS_IN_SPC and N_FILLER > 0:
    add_filler_boundary(
        axes[0], N_FILM, N_INTERFERENCE, N_FILLER, STD_PRESENTED,
        n_break=N_BREAK_SHOWN, show_fillers=True,
    )
axes[0].set_title("SPC by Start Drift Scale", fontsize=16, pad=35)

plot_summary_dv(
    start_drift_scale_values, means, ci_lo, ci_hi,
    xlabel="Start Drift Scale", axis=axes[1],
)
axes[1].set_title("Film Items Recalled", fontsize=16, pad=35)


## Sweeping choice sensitivity (tau) scale

Choice sensitivity (tau) is the exponent applied to M\_CF activations before the Luce choice rule.
At low tau, all items with nonzero activation compete roughly equally — interference items dilute film recall.
At high tau, the choice rule sharpens: retrieval probability concentrates on the most strongly activated items, suppressing weakly activated competitors.

For tau's protective effect to be visible, the retrieval probe must already be biased away from the interference region — otherwise sharpening the choice rule can actually *help* interference items win.
We therefore hold `start_drift_scale` at an elevated value (2.0) and sweep tau from 0.2 to 3.0.

In [ ]:
TAU_SWEEP_START_DRIFT_SCALE = 2.0
tau_scale_values = np.linspace(0.2, 3.0, 10)
tau_recalls_4d, rng = run_sweep(
    std_prep, rng,
    tau_scale=tau_scale_values,
    start_drift_scale=TAU_SWEEP_START_DRIFT_SCALE,
    filler_drift_scale=DEFAULT_FILLER_DRIFT_SCALE,
    filler_mcf_scale=DEFAULT_FILLER_MCF_SCALE,
)

tau_spcs, tau_stats = [], []
for i in range(len(tau_scale_values)):
    recalls = tau_recalls_4d[i].reshape(-1, tau_recalls_4d.shape[-1])
    recalls = standard_remap(recalls, paradigm)
    tau_spcs.append(fixed_pres_spc(recalls, STD_PRESENTED))
    tau_stats.append(film_recalled_stats(recalls, paradigm, n_subjects))

print(f"Tau scale sweep done (start_drift_scale={TAU_SWEEP_START_DRIFT_SCALE}): {tau_scale_values.round(2)}")

In [ ]:
labels = ["{:.2f}".format(v) for v in tau_scale_values]
colors = light_to_dark_colors(len(tau_spcs))
means, ci_lo, ci_hi = zip(*tau_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_interference_spc(
    tau_spcs, labels, n_film=N_FILM, n_break=N_BREAK_SHOWN,
    n_presented=STD_PRESENTED, color_cycle=colors, axis=axes[0],
)
if SHOW_FILLERS_IN_SPC and N_FILLER > 0:
    add_filler_boundary(
        axes[0], N_FILM, N_INTERFERENCE, N_FILLER, STD_PRESENTED,
        n_break=N_BREAK_SHOWN, show_fillers=True,
    )
axes[0].set_title("SPC by Tau Scale", fontsize=16, pad=35)

plot_summary_dv(
    tau_scale_values, means, ci_lo, ci_hi,
    xlabel="Tau Scale", axis=axes[1],
)
axes[1].set_title("Film Items Recalled", fontsize=16, pad=35)


## Interaction: start drift × tau

Do the two retrieval-control mechanisms interact?
A 2×2 comparison crosses low and high values of each scale factor.
If tau adds value beyond start drift alone, the high-tau conditions should show additional film recall recovery even when start drift is already elevated.

In [ ]:
sds_lo, sds_hi = 0.5, 2.5
tau_lo, tau_hi = 0.5, 2.0

interaction_conditions = [
    ("Low start / Low tau", sds_lo, tau_lo),
    ("Low start / High tau", sds_lo, tau_hi),
    ("High start / Low tau", sds_hi, tau_lo),
    ("High start / High tau", sds_hi, tau_hi),
]

interaction_spcs = []
interaction_stats = []

for label, sds, ts in interaction_conditions:
    ready = configure_rates(
        std_prep.models,
        start_drift_scale=sds,
        tau_scale=ts,
        filler_drift_scale=DEFAULT_FILLER_DRIFT_SCALE,
        filler_mcf_scale=DEFAULT_FILLER_MCF_SCALE,
    )
    rngs_2d, rng = sweep_rngs(rng, n_subjects, EXPERIMENT_COUNT)
    recalls_3d = std_prep.batched(ready, rngs_2d, *std_prep.item_args)
    recalls = recalls_3d.reshape(-1, recalls_3d.shape[-1])
    recalls = standard_remap(recalls, paradigm)
    interaction_spcs.append(fixed_pres_spc(recalls, STD_PRESENTED))
    interaction_stats.append(film_recalled_stats(recalls, paradigm, n_subjects))

print("Interaction sweep done")

In [ ]:
labels = [c[0] for c in interaction_conditions]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_interference_spc(
    interaction_spcs, labels, N_FILM, n_break=N_BREAK_SHOWN, axis=axes[0],
)
add_filler_boundary(
    axes[0], N_FILM, N_INTERFERENCE, N_FILLER, STD_PRESENTED, n_break=N_BREAK_SHOWN,
)
axes[0].set_title("SPC: Start Drift \u00d7 Tau Interaction", fontsize=16, pad=20)

# Bar chart for 2x2
means, ci_lo, ci_hi = zip(*interaction_stats)
x_pos = np.arange(len(labels))
errs = [m - lo for m, lo in zip(means, ci_lo)]
axes[1].bar(x_pos, means, yerr=errs, capsize=5, color=["#bbb", "#888", "#666", "#333"])
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=11)
axes[1].set_ylabel("Film Items Recalled", fontsize=14)
axes[1].set_title("Film Items Recalled: 2\u00d72", fontsize=16, pad=20)
for loc in ("top", "right"):
    axes[1].spines[loc].set_visible(False)

plt.tight_layout()
plt.show()

# Print table
print(f"{'Condition':<30s}  {'Mean':>6s}  {'95% CI':>16s}")
print("-" * 56)
for label, (mu, lo, hi) in zip(labels, interaction_stats):
    print(f"{label:<30s}  {mu:6.2f}  [{lo:6.2f}, {hi:6.2f}]")

## Summary

Two retrieval-phase mechanisms modulate vulnerability to context-overlap interference:

1. **Starting-context reinstatement** biases the retrieval probe away from the interference region. At high scales, this shifts the SPC toward early-list items and progressively reduces competition from interference items.

2. **Choice sensitivity (tau)** sharpens the Luce choice rule, concentrating retrieval probability on strongly activated items. Film items with strong primacy-boosted M\_CF associations are protected; weakly activated competitors are suppressed.

Together, these mechanisms provide graded retrieval-control immunity. The 2\u00d72 comparison shows whether their effects are additive or interactive — whether tau adds protection beyond what start-drift reinstatement achieves alone.

**Postdiction**: Intentional free recall is sometimes spared despite using context-to-item retrieval (Lau-Zhu et al. 2019), because retrieval-control mechanisms compensate for encoding-phase interference. Tasks with lower retrieval control (e.g. involuntary intrusions, cued paradigms) remain vulnerable.